In [1]:
%pip install -U --quiet google-genai

In [ ]:
# 인증 및 클라이언트 초기화
import getpass
from google import genai
from google.genai import types
from google import genai
API_KEY = getpass.getpass("Google AI Studio API 키를 입력하세요: ")
client = genai.Client(api_key=API_KEY)

print("✅ Google AI 연결 완료")

In [ ]:
# 어떤 이미지를 만들지 입력 받기
while True:
    choice = input("로고 또는 책 표지 중 무엇을 생성할까요? (로고/책 표지) ").strip().lower()
    if choice in ["로고", "책 표지"]:
        break
    print("로고 또는 책 표지 중에서만 선택해주세요.")

# 2) 선택에 따라 프롬프트 구성
if choice == "로고":
    brand_name = input("브랜드 이름을 입력해주세요: ").strip()
    logo_style = input("로고의 무드/스타일을 입력해주세요 (예: 미니멀, 파스텔, 귀여운): ").strip()

    # 👉 여기서 '텍스트 넣지 마'를 명시적으로 넣어줘야 이상한 한글 안 나옴
    prompt = (
        f"{logo_style} 스타일의 로고, "
        f"브랜드 컨셉은 '{brand_name}', "
        f"한글/영어/숫자등의 텍스트는 넣지 말 것, "
        f"로고 그래픽만 출력"
    )
    aspect_ratio = "1:1"
else:
    book_title = input("책 제목을 입력해주세요: ").strip()
    book_summary = input("책 줄거리를 한 줄로 입력해주세요: ").strip()
    book_mood = input("표지의 분위기를 입력해주세요 (예: 스릴러, 로맨스, 판타지): ").strip()

    prompt = (
        f"{book_mood} 무드의 일러스트 책 표지, "
        f"제목: '{book_title}', 내용: {book_summary}, "
        f"텍스트는 가능한 한 단순하게, 과도한 장식용 글자는 넣지 말 것"
    )
    aspect_ratio = "3:4"

print("📝 최종 프롬프트:", prompt)


In [ ]:
# (셀 4) 이미지 생성하고 Jupyter에서 보기
from io import BytesIO
from PIL import Image
import IPython.display as disp

# 1) 모델 호출
resp = client.models.generate_images(
    model="imagen-4.0-generate-001",
    prompt=prompt,
    config=types.GenerateImagesConfig(
        number_of_images=1,
        aspect_ratio=aspect_ratio,
        image_size="2K",
    ),
)

# 2) 첫 번째 이미지 꺼내기
img_bytes = resp.generated_images[0].image.image_bytes
img = Image.open(BytesIO(img_bytes))

# 3) 노트북에서 너무 크게 보이지 않게 미리보기만 축소
preview = img.copy()
preview.thumbnail((512, 512))  # 발표 화면용
disp.display(preview)

# 4) 원본을 저장해두고 싶으면 아래 주석 해제
# img.save("output.png")